In [ ]:
# allow to sync with local code
%load_ext autoreload
%autoreload 2

In [ ]:
EXPERIMENT_IDS_FULL = {
    "CIFAR_10": "398072040094588659",
    "CIFAR_10 - 10%": "467724091235265108",
    "Pascal_VOC": "563216254920969354",
    "Pascal_VOC - 10%": "570207184802051099",
    "Imdb": "401871667924599083",
    "Imdb - 10%": "298372821182901995",
}
METRICS = ("test_auroc", "test_acc", "test_iou", "test_micro_iou")

SHOW_GOLD_RUNS = True

In [ ]:
from mlflow import MlflowClient

client = MlflowClient(
    tracking_uri="/home/yann/Documents/YCH-perso/goldener/mlflow/mlruns/"
)


def get_metric_for_experiment(experiment_id, metric_names, name_filters=["gold"]):
    """
    Get run name and specific metric value for all runs in an experiment.
    """
    runs = client.search_runs(experiment_ids=[experiment_id])

    run_data = []
    for run in runs:
        run_name = run.info.run_name
        if name_filters is not None:
            if any(name_filter in run_name for name_filter in name_filters):
                continue
        split_random_state = run.data.params.get("random_split_state")
        run_info = {"run_name": run_name, "seed": split_random_state}
        metrics_info = {}
        for metric_name in metric_names:
            metric_value = run.data.metrics.get(metric_name, None)
            if metric_value is not None:
                metrics_info[metric_name] = metric_value
        if metrics_info:
            run_data.append(run_info | metrics_info)

    return run_data


get_metric_for_experiment(
    EXPERIMENT_IDS_FULL["CIFAR_10"],
    METRICS,
    name_filters=[] if SHOW_GOLD_RUNS else ["gold"]
)

In [ ]:
# --- Collect all data for all datasets ---
all_data = {}
for dataset_name, experiment_id in EXPERIMENT_IDS_FULL.items():
    all_data[dataset_name] = get_metric_for_experiment(
        experiment_id,
        METRICS,
        name_filters=[] if SHOW_GOLD_RUNS else ["gold"]
    )

all_data

In [ ]:
import pandas as pd

# --- Convert to flat DataFrame ---
rows = []
for dataset_name, runs in all_data.items():
    for run in runs:
        for metric in METRICS:
            if metric in run:
                rows.append(
                    {
                        "dataset": dataset_name,
                        "run_name": run["run_name"],
                        "seed": str(run["seed"]) if "gold" not in run["run_name"] else "gold",  # str for categorical hue
                        "metric": metric,
                        "value": run[metric],
                    }
                )

df = pd.DataFrame(rows)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Build one global seed->color mapping so each seed keeps the same color everywhere.
seed_order = sorted(df["seed"].dropna().unique(), key=lambda s: (str(s) == "gold", str(s)))
seed_colors = sns.color_palette("tab10", n_colors=max(len(seed_order), 1))
seed_palette = {seed: seed_colors[i % len(seed_colors)] for i, seed in enumerate(seed_order)}
seed_palette["gold"] = "#FFD700"
for dataset in EXPERIMENT_IDS_FULL.keys():
    dataset_df = df[df["dataset"] == dataset]
    dataset_metrics = [m for m in METRICS if m in dataset_df["metric"].values]
    if not dataset_metrics:
        continue
    fig, axes = plt.subplots(
        1,
        len(dataset_metrics),
        figsize=(4 * len(dataset_metrics), 5),
        squeeze=False,
    )
    for j, metric in enumerate(dataset_metrics):
        ax = axes[0][j]
        subset = dataset_df[dataset_df["metric"] == metric]
        # Box plot: shows median, IQR and overall dispersion
        sns.boxplot(
            data=subset,
            y="value",
            ax=ax,
            color="lightsteelblue",
            width=0.35,
            fliersize=0,
            linewidth=1.5,
        )
        # Strip plot: one dot per seed, each seed has a stable color across all plots.
        sns.stripplot(
            data=subset,
            y="value",
            hue="seed",
            hue_order=seed_order,
            ax=ax,
            size=9,
            jitter=0.15,
            alpha=0.85,
            palette=seed_palette,
        )
        ax.set_title(metric, fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("value" if j == 0 else "")
        ax.legend(title="seed", fontsize=7, title_fontsize=8, loc="best")
    fig.suptitle(f"{dataset}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
metrics_present = [m for m in METRICS if m in df["metric"].values]

summary = df.groupby(["dataset", "metric"])["value"].agg(min="min", max="max").round(4)

# Build one row per dataset with "min / max" string per metric
table_rows = []
for dataset in EXPERIMENT_IDS_FULL.keys():
    row = {"Dataset": dataset}
    for metric in metrics_present:
        try:
            mn = summary.loc[(dataset, metric), "min"]
            mx = summary.loc[(dataset, metric), "max"]
            row[metric] = f"{mn} / {mx}"
        except KeyError:
            row[metric] = "-"
    table_rows.append(row)

table_df = pd.DataFrame(table_rows).set_index("Dataset")
print(table_df.to_markdown())

In [ ]:
# create a table showing for each dataset and metric the difference between
# the min and max random seed and the gold seed (if present)
comparison_rows = []
for dataset in EXPERIMENT_IDS_FULL.keys():
    for metric in METRICS:
        subset = df[(df["dataset"] == dataset) & (df["metric"] == metric)]
        if subset.empty:
            continue
        random_values = subset[subset["seed"] != "gold"]["value"]
        gold_values = subset[subset["seed"] == "gold"]["value"]
        random_min = random_values.min() if not random_values.empty else None
        random_max = random_values.max() if not random_values.empty else None
        gold_value = gold_values.iloc[0] if not gold_values.empty else None
        # Signed difference: random value minus gold value.
        diff_min_vs_gold = (
            random_min - gold_value
            if random_min is not None and gold_value is not None
            else None
        )
        diff_max_vs_gold = (
            random_max - gold_value
            if random_max is not None and gold_value is not None
            else None
        )
        comparison_rows.append(
            {
                "dataset": dataset,
                "metric": metric,
                "random_min": random_min,
                "random_max": random_max,
                "gold": gold_value,
                "random_min_minus_gold": diff_min_vs_gold,
                "random_max_minus_gold": diff_max_vs_gold,
            }
        )
comparison_df = pd.DataFrame(comparison_rows).sort_values(["dataset", "metric"]).reset_index(drop=True)
def _fmt(v):
    return "-" if pd.isna(v) else f"{v:.5f}"
display_df = comparison_df.copy()
for col in [
    "random_min",
    "random_max",
    "gold",
    "random_min_minus_gold",
    "random_max_minus_gold",
]:
    display_df[col] = display_df[col].map(_fmt)
print(display_df.to_markdown(index=False))